# 04. ML Baseline Classification

TF-IDF 기반 전통 ML 모델로 민원 텍스트 분류 baseline을 구축합니다.

| Item | Detail |
|------|--------|
| Task | domain(14) / category(63) / intent(1,081) 분류 |
| Models | LogisticRegression, LinearSVC, LightGBM |
| Features | TF-IDF (unigram+bigram, max 50K) |
| Primary Metric | Macro F1 Score |
| Environment | Kaggle T4 x2 GPU |
| Class Balance | class_weight="balanced" |

---
## 0. Environment Setup

In [ ]:
%%capture
!pip install -q plotly kaleido

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'

warnings.filterwarnings('ignore')

# Kaggle vs Local path
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

MODELS_DIR = os.path.join(OUT_DIR, 'models/classification/ml_baseline')
RESULTS_DIR = os.path.join(OUT_DIR, 'results')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data: {DATA_DIR}")
print(f"Models: {MODELS_DIR}")

---
## 1. Data Loading

In [ ]:
train_df = pd.read_parquet(f'{DATA_DIR}/train_classification.parquet')
val_df = pd.read_parquet(f'{DATA_DIR}/val_classification.parquet')
test_df = pd.read_parquet(f'{DATA_DIR}/test_classification.parquet')

with open(f'{DATA_DIR}/class_weights.json') as f:
    class_weights_dict = json.load(f)
with open(f'{DATA_DIR}/label_mapping.json') as f:
    label_mapping = json.load(f)

print(f"Train: {len(train_df):,} / Val: {len(val_df):,} / Test: {len(test_df):,}")
print(f"Columns: {list(train_df.columns)}")
print(f"Domain: {len(label_mapping['domain'])} / Category: {len(label_mapping['category'])} / Intent: {len(label_mapping['intent'])}")
train_df[['text','domain','category','intent_clean']].head(3)

---
## 2. TF-IDF Feature Engineering

In [ ]:
print("=== TF-IDF Vectorizer ===")
t0 = time.time()

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3, max_df=0.95,
    dtype=np.float32,
)

X_train = tfidf.fit_transform(train_df['text'])
X_val = tfidf.transform(val_df['text'])
X_test = tfidf.transform(test_df['text'])

print(f"Vocab: {len(tfidf.vocabulary_):,}")
print(f"Shape: {X_train.shape}")
print(f"Sparsity: {1 - X_train.nnz/(X_train.shape[0]*X_train.shape[1]):.4f}")
print(f"Memory: {X_train.data.nbytes/1e6:.1f} MB")
print(f"Time: {time.time()-t0:.1f}s")

In [ ]:
# Label encoding
label_encoders = {}
y_train, y_val, y_test = {}, {}, {}

for level in ['domain', 'category', 'intent_clean']:
    le = LabelEncoder()
    y_train[level] = le.fit_transform(train_df[level])
    y_val[level] = le.transform(val_df[level])
    y_test[level] = le.transform(test_df[level])
    label_encoders[level] = le
    print(f"{level}: {len(le.classes_)} classes")

---
## 3. Model Training

3 models × 3 levels = 9 experiments

In [ ]:
def train_eval(model, name, X_tr, X_v, y_tr, y_v, level):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    t_train = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(X_v)
    t_infer = (time.time() - t0) / len(y_v) * 1000

    mf1 = f1_score(y_v, y_pred, average='macro', zero_division=0)
    wf1 = f1_score(y_v, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_v, y_pred)

    print(f"  {level:<14} Macro F1={mf1:.4f}  W-F1={wf1:.4f}  Acc={acc:.4f}  "
          f"Train={t_train:.1f}s  Infer={t_infer:.3f}ms")
    return {'model': name, 'level': level,
            'macro_f1': round(mf1,4), 'weighted_f1': round(wf1,4),
            'accuracy': round(acc,4),
            'train_time_s': round(t_train,1), 'infer_ms': round(t_infer,3)}, model

all_results = []
trained_models = {}

### 3.1 Logistic Regression

In [ ]:
print("=== Logistic Regression ===")
for level in ['domain', 'category', 'intent_clean']:
    lr = LogisticRegression(
        max_iter=500, C=1.0,
        class_weight='balanced',
        solver='lbfgs',
        n_jobs=-1, random_state=42,
    )
    r, m = train_eval(lr, 'TF-IDF + LR', X_train, X_val,
                       y_train[level], y_val[level], level)
    all_results.append(r)
    trained_models[f'lr_{level}'] = m

### 3.2 Linear SVC

In [ ]:
print("=== Linear SVC ===")
for level in ['domain', 'category', 'intent_clean']:
    svc = LinearSVC(
        C=1.0, class_weight='balanced',
        max_iter=2000, random_state=42,
    )
    r, m = train_eval(svc, 'TF-IDF + SVC', X_train, X_val,
                       y_train[level], y_val[level], level)
    all_results.append(r)
    trained_models[f'svc_{level}'] = m

### 3.3 LightGBM

In [ ]:
import lightgbm as lgb

print("=== LightGBM ===")
for level in ['domain', 'category', 'intent_clean']:
    lgbm = lgb.LGBMClassifier(
        n_estimators=200, max_depth=8,
        learning_rate=0.1, num_leaves=63,
        class_weight='balanced',
        n_jobs=-1, random_state=42, verbose=-1,
        device='gpu' if IS_KAGGLE else 'cpu',
    )
    r, m = train_eval(lgbm, 'TF-IDF + LGBM', X_train, X_val,
                       y_train[level], y_val[level], level)
    all_results.append(r)
    trained_models[f'lgbm_{level}'] = m

---
## 4. Validation Results

In [ ]:
results_df = pd.DataFrame(all_results)
results_df

In [ ]:
# Macro F1 by level
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Domain (14)', 'Category (63)', 'Intent (1,081)'],
    shared_yaxes=True)

colors = {'TF-IDF + LR':'#42a5f5', 'TF-IDF + SVC':'#66bb6a', 'TF-IDF + LGBM':'#ffa726'}

for ci, level in enumerate(['domain','category','intent_clean'], 1):
    sub = results_df[results_df['level']==level]
    for _, row in sub.iterrows():
        fig.add_trace(go.Bar(
            x=[row['model']], y=[row['macro_f1']],
            marker_color=colors[row['model']],
            text=f"{row['macro_f1']:.4f}", textposition='outside',
            showlegend=(ci==1), name=row['model'], legendgroup=row['model'],
        ), row=1, col=ci)

fig.update_yaxes(title_text='Macro F1', range=[0,1], row=1, col=1)
fig.update_layout(title='ML Baseline: Macro F1 by Level',
                  width=1000, height=450, barmode='group', margin=dict(t=80))
fig.show(renderer='iframe')

In [ ]:
# Speed comparison
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Training Time (s) — Domain', 'Inference (ms/sample) — Domain'])
dr = results_df[results_df['level']=='domain']
c = ['#42a5f5','#66bb6a','#ffa726']

fig.add_trace(go.Bar(x=dr['model'], y=dr['train_time_s'], marker_color=c,
    text=[f"{v:.1f}s" for v in dr['train_time_s']], textposition='outside',
    showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=dr['model'], y=dr['infer_ms'], marker_color=c,
    text=[f"{v:.3f}ms" for v in dr['infer_ms']], textposition='outside',
    showlegend=False), row=1, col=2)

fig.update_layout(width=900, height=400, margin=dict(t=60))
fig.show(renderer='iframe')

---
## 5. Best Model Analysis

In [ ]:
model_key_map = {'TF-IDF + LR':'lr', 'TF-IDF + SVC':'svc', 'TF-IDF + LGBM':'lgbm'}
best_models = {}

print("=== Best Model per Level (Val) ===")
for level in ['domain','category','intent_clean']:
    sub = results_df[results_df['level']==level]
    best = sub.loc[sub['macro_f1'].idxmax()]
    best_models[level] = best['model']
    print(f"  {level:<14} → {best['model']} (Macro F1: {best['macro_f1']:.4f})")

In [ ]:
# Test evaluation — best domain model
bname = best_models['domain']
bmodel = trained_models[f"{model_key_map[bname]}_domain"]
yp = bmodel.predict(X_test)
le_d = label_encoders['domain']

print(f"=== Test Report ({bname} — Domain) ===
")
print(classification_report(y_test['domain'], yp, target_names=le_d.classes_, zero_division=0))

test_macro = f1_score(y_test['domain'], yp, average='macro', zero_division=0)
test_weighted = f1_score(y_test['domain'], yp, average='weighted', zero_division=0)
test_acc = accuracy_score(y_test['domain'], yp)
print(f"Test  Macro F1={test_macro:.4f}  W-F1={test_weighted:.4f}  Acc={test_acc:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test['domain'], yp)
cm_n = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig = go.Figure(go.Heatmap(
    z=cm_n, x=le_d.classes_, y=le_d.classes_, colorscale='Blues',
    text=[[f'{v:.0f}' for v in row] for row in cm],
    texttemplate='%{text}', textfont=dict(size=9),
))
fig.update_layout(title=f'Confusion Matrix — Domain ({bname})',
    xaxis_title='Predicted', yaxis_title='True',
    width=700, height=600, yaxis=dict(autorange='reversed'))
fig.show(renderer='iframe')

In [ ]:
# Per-class F1
pf1 = f1_score(y_test['domain'], yp, average=None, zero_division=0)
f1_df = pd.DataFrame({
    'domain': le_d.classes_, 'f1': pf1,
    'support': np.bincount(y_test['domain'], minlength=len(le_d.classes_))
}).sort_values('f1')

fig = go.Figure(go.Bar(
    y=f1_df['domain'], x=f1_df['f1'], orientation='h',
    marker_color=['#ef5350' if v<0.5 else '#ffa726' if v<0.7 else '#66bb6a' for v in f1_df['f1']],
    text=[f"{v:.3f} (n={s:,})" for v,s in zip(f1_df['f1'], f1_df['support'])],
    textposition='outside',
))
fig.add_vline(x=test_macro, line_dash='dash', line_color='red',
              annotation_text=f'Macro F1: {test_macro:.4f}')
fig.update_layout(title=f'Per-Class F1 — Domain ({bname})',
    xaxis_title='F1', xaxis_range=[0,1.15], width=800, height=500)
fig.show(renderer='iframe')

---
## 6. Save Models & Results

In [ ]:
# Save models
joblib.dump(tfidf, f'{MODELS_DIR}/tfidf_vectorizer.joblib')
joblib.dump(label_encoders, f'{MODELS_DIR}/label_encoders.joblib')

for key, model in trained_models.items():
    joblib.dump(model, f'{MODELS_DIR}/{key}.joblib')

for level in ['domain','category','intent_clean']:
    n = best_models[level]
    k = f"{model_key_map[n]}_{level}"
    joblib.dump(trained_models[k], f'{MODELS_DIR}/best_{level}.joblib')

print(f"Saved {len(trained_models)} models → {MODELS_DIR}")

In [ ]:
# Test all models → JSON
final = {"validation": all_results, "test": [], "best_models": {}}

for level in ['domain','category','intent_clean']:
    for pfx, nm in [('lr','TF-IDF + LR'),('svc','TF-IDF + SVC'),('lgbm','TF-IDF + LGBM')]:
        m = trained_models[f'{pfx}_{level}']
        yp = m.predict(X_test)
        final['test'].append({
            'model': nm, 'level': level,
            'macro_f1': round(f1_score(y_test[level], yp, average='macro', zero_division=0), 4),
            'weighted_f1': round(f1_score(y_test[level], yp, average='weighted', zero_division=0), 4),
            'accuracy': round(accuracy_score(y_test[level], yp), 4),
        })

for level in ['domain','category','intent_clean']:
    sub = [r for r in final['test'] if r['level']==level]
    final['best_models'][level] = max(sub, key=lambda x: x['macro_f1'])

rpath = f'{RESULTS_DIR}/classification_ml_results.json'
with open(rpath, 'w') as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

print(f"Results → {rpath}")
print("
=== Test Results ===")
pd.DataFrame(final['test'])

---
## 7. Summary

In [ ]:
print("="*60)
print("     04. ML Baseline Classification — Summary")
print("="*60)
for level, info in final['best_models'].items():
    print(f"  {level:<14} {info['model']:<18} Macro F1={info['macro_f1']:.4f}")
print(f"
Artifacts → {MODELS_DIR}")
print(f"Results   → {rpath}")
print("
Next → 05_deep_classification (KoBERT/KoELECTRA, Kaggle T4)")
print("="*60)

---
## 8. Base64 다운로드 (Kaggle)

In [ ]:
import base64, io, zipfile
from IPython.display import display, HTML

def create_download_link(filepath, filename=None):
    """파일을 Base64로 인코딩하여 다운로드 링크 생성 (Kaggle용)"""
    if filename is None:
        filename = os.path.basename(filepath)
    if not os.path.exists(filepath):
        print(f"⚠️ 파일 없음: {filepath}")
        return
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / (1024 * 1024)
    href = f'<a href="data:application/octet-stream;base64,{b64}" download="{filename}">📥 {filename} ({size_mb:.1f} MB)</a>'
    display(HTML(href))

def create_zip_download(file_dict, zip_name="artifacts.zip"):
    """여러 파일을 ZIP으로 묶어 Base64 다운로드"""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for arcname, filepath in file_dict.items():
            if os.path.exists(filepath):
                zf.write(filepath, arcname)
            else:
                print(f"⚠️ 스킵: {filepath}")
    buffer.seek(0)
    data = buffer.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / (1024 * 1024)
    href = f'<a href="data:application/octet-stream;base64,{b64}" download="{zip_name}">📦 {zip_name} ({size_mb:.1f} MB)</a>'
    display(HTML(href))

# 결과 JSON 다운로드
create_download_link(f'{RESULTS_DIR}/classification_ml_results.json')

# 모델 아티팩트 ZIP 다운로드
model_files = {}
for fname in os.listdir(MODELS_DIR):
    fpath = os.path.join(MODELS_DIR, fname)
    if os.path.isfile(fpath):
        model_files[f'ml_baseline/{fname}'] = fpath

if model_files:
    create_zip_download(model_files, 'ml_baseline_models.zip')

print("✅ 다운로드 링크 생성 완료")

---
## 9. TF-IDF 하이퍼파라미터 선택 근거

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `max_features` | 50,000 | 한국어 민원 도메인의 어휘 다양성을 커버하면서도 차원의 저주를 방지하는 균형점. 아래 ablation에서 검증 |
| `ngram_range` | (1, 2) | 단일 형태소(unigram)만으로는 "도로 파손" vs "도로 공사"처럼 bigram이 중요한 민원 표현을 구분 불가 |
| `sublinear_tf` | True | 빈도 편향 완화: `tf → 1 + log(tf)`. 민원 텍스트에서 반복 출현하는 "문의", "요청" 등의 과대 가중치 방지 |
| `min_df` | 3 | 오타·희귀 고유명사 제거. 최소 3개 문서에 등장해야 feature로 포함 |
| `max_df` | 0.95 | 95% 이상 문서에 등장하는 범용 어휘("합니다", "있습니다") 제거 — 사실상 불용어 필터 |
| `class_weight` | balanced | 14개 도메인 간 최대 10배 이상 불균형 존재. inverse frequency 가중으로 소수 도메인 학습 보완 |
| `solver` (LR) | lbfgs | 고차원 sparse feature에 적합한 quasi-Newton 방법. L-BFGS는 메모리 효율적이며 large-scale에 수렴 안정적 |

In [ ]:
# === TF-IDF Vocabulary Ablation: 10K / 30K / 50K / 100K ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import plotly.graph_objects as go

vocab_sizes = [10000, 30000, 50000, 100000]
ablation_results = []

for vs in vocab_sizes:
    _tfidf = TfidfVectorizer(
        max_features=vs, ngram_range=(1, 2),
        sublinear_tf=True, min_df=3, max_df=0.95, dtype=np.float32,
    )
    _X_tr = _tfidf.fit_transform(train_df['text'])
    _X_val = _tfidf.transform(val_df['text'])
    _lr = LogisticRegression(max_iter=500, C=1.0, class_weight='balanced',
                             solver='lbfgs', n_jobs=-1, random_state=42)
    _lr.fit(_X_tr, y_train['domain'])
    _pred = _lr.predict(_X_val)
    _mf1 = f1_score(y_val['domain'], _pred, average='macro', zero_division=0)
    ablation_results.append({'vocab': vs, 'macro_f1': round(_mf1, 4)})
    print(f"  max_features={vs:>7,}  →  Domain Macro F1 = {_mf1:.4f}")

abl_df = pd.DataFrame(ablation_results)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=abl_df['vocab'], y=abl_df['macro_f1'],
    mode='lines+markers+text',
    text=[f'{v:.4f}' for v in abl_df['macro_f1']],
    textposition='top center',
    marker=dict(size=12, color='#42a5f5'),
    line=dict(width=3, color='#42a5f5'),
))
fig.add_vline(x=50000, line_dash='dash', line_color='red',
              annotation_text='선택: 50K', annotation_position='top right')
fig.update_layout(
    title='TF-IDF Vocab Size Ablation (Domain, LR, Macro F1)',
    xaxis_title='max_features', yaxis_title='Macro F1',
    xaxis=dict(tickvals=vocab_sizes),
    width=750, height=420,
)
fig.show(renderer='iframe')

---
## 10. Stage Summary — ML Baseline → Deep Classification

**이 단계에서 확인한 것:**
- TF-IDF + Linear Model은 강력한 baseline: Domain 분류에서 Macro F1 ≈ 0.65–0.70 수준 달성
- Linear SVC가 대부분의 레벨에서 LR/LGBM 대비 소폭 우위
- **한계**: TF-IDF는 어순/문맥 정보를 잃어버림 → "소음 민원 접수" vs "접수된 소음 민원" 구분 불가

**다음 단계 (05_deep_classification) 기대 효과:**
- KoBERT/KoELECTRA는 양방향 문맥(contextualized embedding)을 학습하여 동일 단어도 문맥에 따라 다른 표현 생성
- 사전학습된 한국어 지식 활용 → 소수 도메인 일반화 능력 향상 기대
- **목표**: TF-IDF baseline 대비 Domain Macro F1 +5~10%p 개선

In [ ]:
# === 도메인별 F1 vs 학습 샘플 수 관계 분석 ===
train_support = np.bincount(y_train['domain'], minlength=len(le_d.classes_))
test_support = np.bincount(y_test['domain'], minlength=len(le_d.classes_))

scatter_df = pd.DataFrame({
    'domain': le_d.classes_,
    'train_count': train_support,
    'test_f1': pf1,
    'test_support': test_support,
})

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=scatter_df['train_count'],
    y=scatter_df['test_f1'],
    mode='markers+text',
    text=scatter_df['domain'],
    textposition='top center',
    textfont=dict(size=9),
    marker=dict(
        size=np.clip(scatter_df['test_support'] / scatter_df['test_support'].max() * 40 + 10, 10, 50),
        color=scatter_df['test_f1'],
        colorscale='RdYlGn',
        cmin=0, cmax=1,
        showscale=True,
        colorbar=dict(title='F1'),
        line=dict(width=1, color='white'),
    ),
    showlegend=False,
))

fig.add_hline(y=test_macro, line_dash='dash', line_color='red',
              annotation_text=f'Macro F1: {test_macro:.4f}')

fig.update_layout(
    title=f'도메인별 F1 vs 학습 샘플 수 ({bname})',
    xaxis_title='학습 샘플 수 (로그 스케일)',
    yaxis_title='테스트 F1 Score',
    xaxis_type='log',
    yaxis_range=[0, 1.1],
    width=850, height=550,
)
fig.show(renderer='iframe')

# Spearman 순위 상관계수 (scipy 의존성 없이 직접 계산)
log_counts = np.log1p(scatter_df['train_count'].values)
f1_vals = scatter_df['test_f1'].values
rank_count = np.argsort(np.argsort(log_counts)).astype(float)
rank_f1 = np.argsort(np.argsort(f1_vals)).astype(float)
n = len(rank_count)
spearman_rho = 1 - 6 * np.sum((rank_count - rank_f1)**2) / (n * (n**2 - 1))

print(f"Spearman rho (학습 샘플 수 vs F1): {spearman_rho:.3f}")
print(f"\n저성능 도메인 (F1 < Macro F1 = {test_macro:.4f}):")
low_f1 = scatter_df[scatter_df['test_f1'] < test_macro].sort_values('test_f1')
for _, row in low_f1.iterrows():
    print(f"  {row['domain']:<15} F1={row['test_f1']:.3f}  학습 샘플={row['train_count']:,}")

---
## 11. 오분류 패턴 분석

### 학습 샘플 수 vs F1 Score 관계

위 scatter plot에서 **로그 스케일의 학습 샘플 수와 F1 Score** 사이에 양의 상관관계가 관찰됩니다.

**저성능 도메인 원인 분석:**

1. **소수 클래스 문제**: `부동산`, `숙박` 등 학습 샘플이 수백 건에 불과한 도메인은 TF-IDF feature space에서 충분한 decision boundary를 형성하지 못함
2. **유사 도메인 혼동**: `부동산` vs `부동산업`은 텍스트 표면형이 거의 동일하여 TF-IDF 기반으로 구분이 어려움 → Confusion Matrix에서 상호 오분류 확인 가능
3. **class_weight의 한계**: `class_weight='balanced'`가 소수 클래스의 loss를 증가시키지만, feature 자체의 discriminative power가 부족하면 개선에 한계가 있음

**시사점:**
- 소수 도메인 성능 개선을 위해서는 **문맥 정보를 활용하는 DL 모델**(KoBERT, KoELECTRA)이 필요
- TF-IDF는 "부동산"이라는 동일 키워드를 공유하는 도메인을 구분할 수 없지만, Transformer는 주변 문맥으로 구분 가능

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
# 다음 노트북(05)에서 ML baseline 결과를 입력으로 사용
if IS_KAGGLE:
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 심볼릭 링크로 메모리 절약 (파일 복사 대신)
    upload_targets = {
        RESULTS_DIR: ['classification_ml_results.json'],
        MODELS_DIR: None,  # 전체 디렉토리
    }

    # 결과 JSON
    for src_dir, files in upload_targets.items():
        if files is None:
            # 디렉토리 내 모든 파일
            if os.path.exists(src_dir):
                for fname in os.listdir(src_dir):
                    src = os.path.join(src_dir, fname)
                    dst = os.path.join(UPLOAD_DIR, fname)
                    if os.path.isfile(src) and not os.path.exists(dst):
                        os.symlink(src, dst)
        else:
            for fname in files:
                src = os.path.join(src_dir, fname)
                dst = os.path.join(UPLOAD_DIR, fname)
                if os.path.exists(src) and not os.path.exists(dst):
                    os.symlink(src, dst)

    # metadata 생성
    meta = {
        "title": "civilcomplaint-ml-baseline",
        "id": "kukass/civilcomplaint-ml-baseline",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(f'{UPLOAD_DIR}/dataset-metadata.json', 'w') as f:
        json.dump(meta, f)

    print(f"업로드 대상: {os.listdir(UPLOAD_DIR)}")
    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("Kaggle 데이터셋 업로드 완료: civilcomplaint-ml-baseline")
else:
    print("로컬 환경 — Kaggle 업로드 생략")